In [ ]:
import io
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import numpy as np

# Crear componentes de la interfaz de ipywidgets
boton_dataset = widgets.Button(
    description="Probar imagen del dataset",
    button_style="info",
    layout=widgets.Layout(width="250px", height="40px")
)

cargador_archivo = widgets.FileUpload(
    accept=".png,.jpg,.jpeg,.bmp",
    multiple=False,
    description="Cargar imagen externa",
    button_style="success",
    layout=widgets.Layout(width="250px", height="40px")
)

salida_grafica = widgets.Output()

# Función para procesar y mostrar resultados de manera visual
def mostrar_resultado_notebook(imagen_28x28):
    # Obtener predicciones
    predicciones, mejores = clasificar_imagen(imagen_28x28)

    with salida_grafica:
        clear_output(wait=True)

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

        # 1. Visualizar imagen
        ax1.imshow(imagen_28x28, cmap="gray")
        ax1.axis("off")
        ax1.set_title("Vista previa de la prenda")

        # 2. Gráfico de barras con probabilidades
        top_categorias = [CATEGORIAS[i] for i in mejores]
        top_valores = [predicciones[i] * 100 for i in mejores]

        colores = ["#62D9A5", "#2878B5", "#BFC9D5"]
        barras = ax2.barh(top_categorias[::-1], top_valores[::-1], color=colores[::-1])
        ax2.set_xlim(0, 100)
        ax2.set_xlabel("Confianza (%)")
        ax2.set_title("Predicción del Modelo")

        # Agregar etiquetas de porcentaje en las barras
        for barra in barras:
            ancho = barra.get_width()
            ax2.text(ancho + 1, barra.get_y() + barra.get_height()/2, f"{ancho:.1f}%",
                     va='center', ha='left', fontweight='bold')

        plt.tight_layout()
        plt.show()

# Acción: Probar imagen aleatoria
def al_presionar_dataset(b):
    indice = np.random.randint(0, len(x_test))
    imagen = x_test[indice].squeeze()
    mostrar_resultado_notebook(imagen)

# Acción: Cargar imagen externa
def al_cargar_archivo(change):
    if not cargador_archivo.value:
        return

    try:
        # Extraer el contenido de la imagen de manera segura según la versión de ipywidgets
        datos_carga = cargador_archivo.value

        # En ipywidgets v8+, es una tupla o lista de diccionarios
        if isinstance(datos_carga, (tuple, list)):
            archivo_info = datos_carga[0]
            contenido = archivo_info['content']
        # En versiones anteriores, es un diccionario donde la clave es el nombre del archivo
        elif isinstance(datos_carga, dict):
            nombre_archivo = list(datos_carga.keys())[0]
            contenido = datos_carga[nombre_archivo]['content']
        else:
            raise ValueError("Estructura de archivo cargado no soportada.")

        # Convertir a escala de grises y redimensionar a 28x28
        imagen = Image.open(io.BytesIO(bytes(contenido))).convert("L")
        imagen = ImageOps.fit(imagen, (28, 28))
        imagen_np = np.array(imagen)

        mostrar_resultado_notebook(imagen_np)
    except Exception as e:
        with salida_grafica:
            clear_output()
            print(f"Error al procesar la imagen: {e}")

    # Resetear el cargador para permitir subir archivos repetidos
    cargador_archivo.value = ()

# Conectar eventos
boton_dataset.on_click(al_presionar_dataset)
cargador_archivo.observe(al_cargar_archivo, names='value')

# Mostrar la interfaz en el notebook
print("--- INTERFAZ INTERACTIVA PARA COLAB ---")
ui = widgets.HBox([boton_dataset, cargador_archivo])
display(ui)
display(salida_grafica)

--- INTERFAZ INTERACTIVA PARA COLAB ---


Output()

TraitError: The "value" trait is read-only.

TraitError: The "value" trait is read-only.